# Module 3 — Topic 4: Seaborn for Statistical Visualization
## Live Demo Notebook

**Scenario:** Same Nigerian salary dataset from Topic 2.
This time: build statistical charts that reveal more than the raw numbers did.

---

**My EDA questions:**
1. What does the salary distribution look like — shape, spread, typical value?
2. Which sectors pay the most — and which have the most variable pay?
3. Does salary increase with experience, and does this differ by sector?
4. Are there correlations between our numeric variables?

---

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Apply a consistent visual theme to all charts in this notebook
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('nigerian_salaries.csv')
print('Shape:', df.shape)
df.head(3)

---
## Part 1 — Matplotlib vs Seaborn: The Same Histogram

In [ ]:
# Matplotlib version — more lines, no KDE
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['salary_monthly'], bins=30, color='mediumpurple', edgecolor='white', alpha=0.85)
ax.set_title('Salary Distribution — Matplotlib')
ax.set_xlabel('Monthly Salary (₦)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn version — fewer lines, plus KDE curve automatically
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=df, x='salary_monthly', bins=30, kde=True, color='mediumpurple', ax=ax)
ax.set_title('Salary Distribution — Seaborn (with KDE)')
ax.set_xlabel('Monthly Salary (₦)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('Same data, less code, plus a smooth density curve.')

---
## Part 2 — `histplot` with Mean and Median

Q1: What does the salary distribution look like?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

sns.histplot(data=df, x='salary_monthly', bins=30, kde=True, color='mediumpurple', ax=ax)

mean_s   = df['salary_monthly'].mean()
median_s = df['salary_monthly'].median()

ax.axvline(mean_s,   color='red',    linestyle='--', linewidth=2,
           label=f'Mean   ₦{mean_s:,.0f}')
ax.axvline(median_s, color='orange', linestyle='--', linewidth=2,
           label=f'Median ₦{median_s:,.0f}')

ax.set_title('Distribution of Nigerian Professional Monthly Salaries', fontsize=14, fontweight='bold')
ax.set_xlabel('Monthly Salary (₦)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Skewness: {df["salary_monthly"].skew():.2f} — right-skewed')
print(f'The mean is ₦{mean_s - median_s:,.0f} higher than the median.')

---
## Part 3 — `boxplot`: Salary by Sector

Q2: Which sectors pay the most — and which have the most variable pay?

In [ ]:
# Order sectors by median salary (highest to lowest)
sector_order = (
    df.groupby('sector')['salary_monthly']
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(13, 5))

sns.boxplot(data=df, x='sector', y='salary_monthly', order=sector_order,
            palette='muted', ax=ax)

ax.set_title('Monthly Salary Distribution by Sector (ordered by median)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sector', fontsize=12)
ax.set_ylabel('Monthly Salary (₦)', fontsize=12)
ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

print('Box anatomy: box = IQR (Q1 to Q3), line = median, whiskers = 1.5×IQR, dots = outliers')

In [ ]:
# Read what the chart tells us numerically
sector_stats = df.groupby('sector')['salary_monthly'].agg(['median', 'std']).sort_values('median', ascending=False)
sector_stats.columns = ['median_salary', 'std_salary']
sector_stats['median_fmt'] = sector_stats['median_salary'].apply(lambda x: f'NGN {x:,.0f}')
sector_stats['std_fmt']    = sector_stats['std_salary'].apply(lambda x: f'NGN {x:,.0f}')
print('Sector salary summary (sorted by median):')
print(sector_stats[['median_fmt', 'std_fmt']].rename(columns={'median_fmt': 'Median', 'std_fmt': 'Std Dev'}))

---
## Part 4 — `scatterplot` with `hue`

Q3: Does salary increase with experience — and does this differ by sector?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.scatterplot(data=df, x='years_experience', y='salary_monthly',
                hue='sector', alpha=0.75, s=65, ax=ax)

ax.set_title('Years of Experience vs Monthly Salary by Sector', fontsize=14, fontweight='bold')
ax.set_xlabel('Years of Experience', fontsize=12)
ax.set_ylabel('Monthly Salary (₦)', fontsize=12)
ax.legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

# Which sector has the steepest experience-salary slope?
print('Correlation (experience vs salary) per sector:')
for sector in df['sector'].unique():
    subset = df[df['sector'] == sector]
    corr   = subset['years_experience'].corr(subset['salary_monthly'])
    print(f'  {sector:<25} {corr:+.3f}')

---
## Part 5 — `barplot` and `countplot`

In [ ]:
# Average salary by sector with standard deviation error bars
sector_mean_order = df.groupby('sector')['salary_monthly'].mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(13, 5))

sns.barplot(data=df, x='sector', y='salary_monthly',
            order=sector_mean_order, palette='muted', ax=ax, errorbar='sd')

ax.set_title('Average Monthly Salary by Sector (bars = mean, lines = ±1 std dev)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sector', fontsize=12)
ax.set_ylabel('Monthly Salary (₦)', fontsize=12)
ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
# How many professionals per sector — class imbalance check
fig, ax = plt.subplots(figsize=(13, 4))

sns.countplot(data=df, x='sector',
              order=df['sector'].value_counts().index, palette='muted', ax=ax)

ax.set_title('Number of Professionals per Sector (class imbalance check)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sector', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

---
## Part 6 — `heatmap` and `pairplot`

Q4: Are there correlations between our numeric variables?

In [ ]:
numeric_cols = ['years_experience', 'salary_monthly']
corr_matrix  = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 4))

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)

ax.set_title('Correlation Matrix — Nigerian Salary Data', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Red = positive correlation | Blue = negative | Diagonal always = 1.0')

In [ ]:
# pairplot: all pairwise relationships at once
pair_df = df[['years_experience', 'salary_monthly', 'gender']].copy()

g = sns.pairplot(pair_df, hue='gender', diag_kind='kde',
                 plot_kws={'alpha': 0.5, 's': 30}, palette='muted')
g.fig.suptitle('Pairplot — Experience & Salary by Gender', y=1.02, fontsize=13)
plt.show()

print('Diagonal: distribution of each variable')
print('Off-diagonal: relationship between each pair')

---
## Part 7 — EDA Dashboard: Four Charts in One Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top-left: salary distribution
sns.histplot(data=df, x='salary_monthly', bins=30, kde=True,
             color='mediumpurple', ax=axes[0, 0])
axes[0, 0].axvline(df['salary_monthly'].median(), color='orange', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('Salary Distribution')
axes[0, 0].set_xlabel('Monthly Salary (₦)')

# Top-right: salary by sector (box)
sns.boxplot(data=df, x='sector', y='salary_monthly', palette='muted',
            order=sector_order, ax=axes[0, 1])
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set_title('Salary by Sector')
axes[0, 1].set_xlabel('')

# Bottom-left: experience vs salary
sns.scatterplot(data=df, x='years_experience', y='salary_monthly',
                hue='sector', alpha=0.6, s=40, ax=axes[1, 0], legend=False)
axes[1, 0].set_title('Experience vs Salary')
axes[1, 0].set_xlabel('Years of Experience')
axes[1, 0].set_ylabel('Monthly Salary (₦)')

# Bottom-right: count by sector
sns.countplot(data=df, x='sector',
              order=df['sector'].value_counts().index, palette='muted', ax=axes[1, 1])
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_title('Records per Sector')
axes[1, 1].set_xlabel('')

fig.suptitle('Nigerian Professional Salary — EDA Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

In this demo we built:
- **`histplot` with KDE** — salary distribution shape + mean/median reference lines
- **`boxplot`** — IQR, median, and outliers per sector (sorted by median)
- **`scatterplot` with `hue`** — experience vs salary coloured by sector
- **`barplot`** — average salary per sector with standard deviation error bars
- **`countplot`** — class imbalance check across sectors
- **`heatmap`** — correlation matrix between numeric variables
- **`pairplot`** — all pairwise relationships in one grid by gender
- **EDA Dashboard** — four charts combined into one figure

**Next — Topic 5:** Choosing the right chart. We explore a framework for chart selection and demonstrate the most common chart-choice mistakes.